# Submit replica chain calculation

In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
# General imports.
import ipywidgets as ipw
from IPython.display import clear_output

# AiiDA imports.
%load_ext aiida
%aiida

# AiiDAlab imports.
import aiidalab_widgets_base as awb
import aiidalab_widgets_empa as awe
from aiida import orm, plugins

from surfaces_tools.utils import wfn

# Custom imports.
from surfaces_tools.widgets import auto_representations, dft_settings, editors, inputs


In [ ]:
Cp2kReplicaWorkChain = plugins.WorkflowFactory("nanotech_empa.cp2k.replica")

In [ ]:
# Structure selector.
build_slab = editors.BuildSlab(title="Build slab")
input_details = inputs.InputDetails()

structure_selector = awb.StructureManagerWidget(
    importers=[
        awb.StructureUploadWidget(title="Import from computer"),
        awb.StructureBrowserWidget(title="AiiDA database"),
        awb.SmilesWidget(title="From SMILES"),
        awe.CdxmlUploadWidget(title="CDXML"),
    ],
    editors=[awb.BasicStructureEditor(title="Edit structure"), build_slab],
    storable=False,
    node_class="StructureData",
)

ipw.dlink((structure_selector, "structure"), (build_slab, "molecule"))
ipw.dlink((structure_selector, "structure"), (input_details, "structure"))
ipw.dlink((input_details, "details"), (build_slab, "details"))
input_details.replica = True
input_details.neb = False

auto_representation = auto_representations.AutoRepresentationWidget(
    structure_selector.viewer
)
ipw.dlink((structure_selector, "structure"), (auto_representation, "structure"))
ipw.dlink((input_details, "details"), (auto_representation, "details"))
display(structure_selector, auto_representation)

# Code.
code_input_widget = awb.ComputationalResourcesWidget(
    description="CP2K code:", default_calc_job_plugin="cp2k"
)
resources = awe.ProcessResourcesWidget()

# Protocol.
protocol = ipw.Dropdown(
    value="standard",
    options=[
        ("Standard", "standard"),
        ("Low accuracy", "low_accuracy"),
        ("Debug", "debug"),
    ],
    description="Protocol:",
    style={"description_width": "initial"},
)

dft_settings_widget = dft_settings.DftSettingsWidget()

advanced_settings = ipw.Accordion(
    children=[dft_settings_widget],
    selected_index=None,
)
advanced_settings.set_title(0, "DFT settings")

output = ipw.Output()


In [ ]:
workflow_description = ipw.Text(
    description="Workflow description:",
    placeholder="Provide the description here.",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)


In [ ]:
ipw.dlink((code_input_widget, "value"), (input_details, "selected_code"))


def prepare_inputs():
    with output:
        clear_output()
    if not structure_selector.structure_node:
        can_submit, msg = False, "Select a structure first."
    elif not code_input_widget.value:
        can_submit, msg = False, "Select CP2K code."
    else:
        can_submit, msg, parameters = input_details.return_final_dictionary()

    if not can_submit:
        with output:
            print(msg)
            return

    try:
        dft_settings_widget.update_dft_params(parameters["dft_params"])
    except ValueError as exc:
        with output:
            print(exc)
        return
    dft_params_dict = parameters["dft_params"]

    builder = Cp2kReplicaWorkChain.get_builder()
    builder.protocol = orm.Str(protocol.value)
    builder.metadata.label = "CP2K_Replica"
    builder.metadata.description = workflow_description.value
    code = orm.load_code(code_input_widget.value)
    builder.code = code
    builder.options = {
        "max_wallclock_seconds": resources.walltime_seconds,
        "resources": {
            "num_machines": resources.nodes,
            "num_mpiprocs_per_machine": resources.tasks_per_node,
            "num_cores_per_mpiproc": resources.threads_per_task,
        },
    }

    builder.structure = structure_selector.structure_node
    builder.dft_params = orm.Dict(dft_params_dict)
    builder.sys_params = orm.Dict(parameters["sys_params"])

    if "restart_from" in parameters:
        builder.restart_from = orm.Str(parameters["restart_from"])
    else:
        # Check if a restart wfn is available for the replica chain calculations not restarting from a previous calculation.
        wave_function = None
        if structure_selector.structure_node.is_stored:
            wave_function = wfn.structure_available_wfn(
                node=structure_selector.structure_node,
                relative_replica_id=None,
                current_hostname=code.computer.hostname,
                return_path=False,
                dft_params=dft_params_dict,
            )
        if wave_function is not None:
            print(f"Restarting from wfn in folder: {wave_function.pk}")
            builder.parent_calc_folder = wave_function

    return builder


In [ ]:
btn_submit = awb.SubmitButtonWidget(
    Cp2kReplicaWorkChain,
    inputs_generator=prepare_inputs,
    disable_after_submit=False,
    append_output=True,
)


In [ ]:
# Resources estimation.
resources_estimation_button = awe.ResourcesEstimatorWidget(
    price_link="https://2go.cscs.ch/offering/swiss_academia/institutional_customers/",
    price_per_hour=2.85,
)
resources_estimation_button.link_to_resources_widget(resources)
ipw.dlink((structure_selector, "structure"), (input_details, "structure"))
ipw.dlink((input_details, "details"), (resources_estimation_button, "details"))
ipw.dlink((input_details, "uks"), (resources_estimation_button, "uks"))
_ = ipw.dlink(
    (code_input_widget, "value"), (resources_estimation_button, "selected_code")
)
input_details.replica = True
input_details.neb = False


# Inputs

In [ ]:
display(protocol, input_details, advanced_settings)

# Code and resources

In [ ]:
display(code_input_widget, ipw.HBox([resources, resources_estimation_button]))

# Submit

In [ ]:
display(workflow_description, btn_submit, output)